In [1]:
###### OSVAS ########################################################
###### (OFFLINE SURFEX VALIDATION SYSTEM)###########################
###### STEP 1: Downloading forcing data from CABAUW #############
#### STEP 1.0: DEFINING STATION and OSVAS PATH ###############
import os
# Default values defined in the notebook for Station and OSVAS install:
OSVAS='/home/pn56/OSVASgh/'  # Main OSVAS path
OSVAS='/home/alvaro/master/TFM/OSVAS'
Station_name='Cabauw'

# If an environment variable STATION or OSVAS exists, override the default
#Station_name = os.getenv("STATION_NAME", Station_name)
OSVAS = os.getenv("OSVAS", OSVAS)

print(f"Creating forcing for {Station_name} with OSVAS installation in {OSVAS}" )

Creating forcing for Cabauw with OSVAS installation in /home/alvaro/master/TFM/OSVAS


In [2]:
###### OSVAS ########################################################
###### ( OFFLINE SURFEX VALIDATION SYSTEM)###########################
#### STEP 1.1: IMPORTING NEEDED PACKAGES AND DEFINING FUNCTIONS ###############
from datetime import date, datetime, timedelta, timezone
from dateutil.relativedelta import relativedelta
import matplotlib
import matplotlib.pyplot as plt
from netCDF4 import Dataset, date2num
import numpy as np
import os
import pandas as pd
import re
import requests
import tempfile
import xarray as xr
import xml.etree.ElementTree as ET
import yaml

##############################################################################
##Here comes a series of functions for handling the forcing creation easily ##
##############################################################################

def apply_transformation(variable, op, val):
    if op == "+":
        return variable + float(val)
    elif op == "-":
        return variable - float(val)
    elif op == "*":
        return variable * float(val)
    elif op == "/":
        return variable / float(val)
    else:
        return variable  # No op

def datespan(startDate, endDate, delta=timedelta(days=1)):
    currentDate = startDate
    while currentDate < endDate:
        yield currentDate
        currentDate += delta

def download_file(endpoint, headers, filename, download_directory='./.nc_data'):
    """
        Downloads the file filename from the endpoint to download_directory. If open is set to true, 
        the function returns the opened Netcdf Dataset.

        Parameters:
            endpoint: The url of the endpoint to receive the list of files
            headers: Dictionary with request headers, Authentication is mandatory with the corresponding token.
            download_directory: The path to the directory where files will be downloaded.
    """
    result = requests.get('/'.join((endpoint, filename, 'url')), headers=headers)
    r = requests.get(result.json()['temporaryDownloadUrl'])
    r.raise_for_status()
    with tempfile.NamedTemporaryFile(suffix=".nc") as f:
        f.write(r.content)
        f.flush()
        ds = xr.open_dataset(f.name)  # or "h5netcdf"
    return ds

def fetch_file_list(endpoint, headers, payload):
    """
        File list returned by endpoint satisfying payload criteria.
        
        Parameters:
            endpoint: The url of the endpoint to receive the list of files
            headers: Dictionary with request headers, Authentication is mandatory with the corresponding token
            payload: Dictionary with optional parameters for the get request.
        
        return: A list with the filenames that satisfy the given criteria. This filenames can then be retrieved calling another KMNI API endpoint
        
    """
    file_list = []
    truncated = True
    while truncated:
        result = requests.get(endpoint, headers=headers, params=payload)
        result.raise_for_status()
        result = result.json()
        file_list += [i['filename'] for i in result['files']]
        truncated = result['isTruncated']
        if truncated:
            payload += {'nextPageToken': result['nextPageToken']}
    print(f"{len(file_list)} files will be processed")
    return file_list

def initialize_forcing_dataset(start_date, end_date, timedelta, station_data):
    time = pd.date_range(start=start_date, end=end_date, freq=f"{timedelta}min").tz_convert("UTC").tz_localize(None)
    forcing_dataset = xr.Dataset(
        coords = {
            "time": ("time", time,),},
        data_vars = {
            "FRC_TIME_STP": (("Number_of_points",), [common_timedelta*60.], {"units": "s", "description": "forcing time step"}),
            "LAT": (("Number_of_points",), [float(station_data['lat'])], {"description": "latitudes" , "units": "degrees"}),
            "LON": (("Number_of_points",), [float(station_data['lon'])], {"description": "longitudes", "units": "degrees"}),
            "ZS": (("Number_of_points",), [float(station_data['elev'])], {"description": "surface orography", "units": "m"}),
            "UREF": (("Number_of_points",), [float(station_data['height_V'])], {"description": "Reference_Height_for_Wind", "units": "m"}),
            "ZREF": (("Number_of_points",), [float(station_data['height_T'])], {"description": "Reference_Height", "units": "m"}),
            "Tair": (("time", "Number_of_points"), np.empty((len(time), 1)), {"description": "air temperature", "units": "K", "ascii_name": "Forc_TA"}),
            "Qair": (("time", "Number_of_points"), np.empty((len(time), 1)), {"description": "air specific humidity", "units": "Kg/Kg", "ascii_name": "Forc_QA"}),
            "Wind": (("time", "Number_of_points"), np.empty((len(time), 1)), {"description": "wind speed", "units": "m/s", "ascii_name": "Forc_WIND"}),
            "DIR_SWdown": (("time", "Number_of_points"), np.empty((len(time), 1)), {"description": "downward direct shortwave radiation", "units": "W/m2", "ascii_name": "Forc_DIR_SW"}),
            "SCA_SWdown": (("time", "Number_of_points"), np.empty((len(time), 1)), {"description": "downward diffuse shortwave radiation", "units": "W/m2", "ascii_name": "Forc_SCA_SW"}),
            "LWdown": (("time", "Number_of_points"), np.empty((len(time), 1)), {"description": "downward longwave radiation", "units": "W/m2", "ascii_name": "Forc_LW"}),
            "PSurf": (("time", "Number_of_points"), np.empty((len(time), 1)), {"description": "surface pressure", "units": "Pa", "ascii_name": "Forc_PS"}),
            "Rainf": (("time", "Number_of_points"), np.empty((len(time), 1)), {"description": "rainfall rate", "units": "Kg/m2/s", "ascii_name": "Forc_RAIN"}),
            "Snowf": (("time", "Number_of_points"), np.empty((len(time), 1)), {"description": "snowfall rate", "units": "Kg/m2/s", "ascii_name": "Forc_SNOW"}),
            "CO2air": (("time", "Number_of_points"), np.empty((len(time), 1)), {"description": "CO2 concentration", "units": "Kg/m3", "ascii_name": "Forc_CO2"}),
            "Wind_DIR": (("time", "Number_of_points"), np.empty((len(time), 1)), {"description": "wind direction", "units": "deg", "ascii_name": "Forc_DIR"}),
    })
    forcing_dataset.time.encoding.update({
        'units': "seconds since 2014-01-01 00:00:00", 
        'calendar': 'gregorian',
        "dtype": "float64"})
    return forcing_dataset

def parse_variable_entry(entry_str, timedelta_minutes):
    if entry_str is None or entry_str.strip() == "":
        return None, "const", 0.0

    parts = [p.strip() for p in entry_str.split(",", maxsplit=1)]

    # Case: constant only
    if len(parts) == 1:
        val = parts[0]
        if val in ["-", "None", ""]:
            return None, "const", 0.0
        try:
            return None, "const", float(val)
        except ValueError:
            return val, None, None  # just a direct mapping

    # Case: transformation
    src, transform = parts
    if src in ["-", "None", ""]:
        try:
            return None, "const", float(transform)
        except ValueError:
            raise ValueError(f"Invalid constant value in entry: {entry_str}")

    if "timedelta" in transform:
        transform = transform.replace("timedelta", str(timedelta_minutes))

    op = transform[0]
    expr = transform[1:].strip()

    try:
        val = eval(expr, {}, {})  # safe eval of math expression
    except Exception as e:
        raise ValueError(f"Failed to evaluate expression '{expr}' in entry '{entry_str}': {e}")

    return src, op, val

def retrieve_data(endpoint, start_date, end_date, key):
    """
    Generates a dataset with the data collected from endpoint between start_date and end_date. It requires the api_key parameter 
    to make the requests to the service.
    """
    dataset_name = endpoint.split("/datasets/")[1].split("/")[0]
    dataset_version = endpoint.split("/versions/")[1].split("/")[0]
    start_file = (f"{dataset_name}_{dataset_version}_{start_date:%Y%m}.nc")
    end_file = (f"{dataset_name}_{dataset_version}_{end_date:%Y%m}.nc")
    months = pd.date_range(start=start_date, end=end_date, freq="MS")
    filenames = [f"{dataset_name}_{dataset_version}_{date:%Y%m}.nc" for date in months]
    headers={"Authorization": api_key}
    payload = {
        'maxKeys': '1000',
        'sorting': 'asc',
        'orderBy': 'filename',
        'begin': start_file,
        'end': end_file
    }
    file_list = fetch_file_list(endpoint, headers, payload)
    files = []
    for file in filenames:
        if file in file_list:
            files.append(download_file(endpoint, headers, file).drop_vars('valid_dates'))
        else:
            print(f"⚠️ {file} not found in KMNI service, data for this month will be filled with nan")
    ds_values = xr.concat(files, dim='time', data_vars='minimal').sortby('time')
    ds_values = ds_values.sel(time=slice(start_date.tz_localize(None), end_date.tz_localize(None)))
    return ds_values

def write_params_config(forcing_path, forcing_dataset):
    ddtt = pd.Timestamp(forcing_dataset['time'].values[0])
    
    param_lines = [
        len(forcing_dataset['LAT']),
        len(forcing_dataset['time']),
        forcing_dataset['FRC_TIME_STP'].values[0],
        ddtt.year,
        ddtt.month,
        ddtt.day,
        ddtt.hour*3600,
        "\t".join(map(str, forcing_dataset['LON'].values)),
        "\t".join(map(str, forcing_dataset['LAT'].values)),
        "\t".join(map(str, forcing_dataset['ZS'].values)),
        "\t".join(map(str, forcing_dataset['ZREF'].values)),
        "\t".join(map(str, forcing_dataset['UREF'].values))
    ]
    with open(os.path.join(forcing_path, "Params_config.txt"), 'w') as f:
                f.write("\n".join(map(str, param_lines)) + "\n")

def write_forcing_ascii(forcing_path, forcing_dataset):
    for var_name, var in forcing_dataset.data_vars.items():
        if "ascii_name" in var.attrs:
            file = var.attrs['ascii_name']
            np.savetxt(f"{os.path.join(forcing_path, file)}.txt", var.values, delimiter='\t', fmt='%.6f')

In [3]:
###### OSVAS #####################################################
###### ( OFFLINE SURFEX VALIDATION SYSTEM)########################
#### STEP 1.2: LOAD STATION METADATA AND CONFIGURATION OF  #########
#### THE FORCING GENERATION FROM THE STATION'S YAML FILE #########

#3.1 Read YAML config for station, define paths, set Station,
#    get station metadata

write_forcing='yes' #Set to yes to write forcing
CONFIG_PATH = os.path.join(OSVAS, "config_files", "Stations", f"{Station_name}.yml")

with open(CONFIG_PATH, "r") as f:
    config = yaml.safe_load(f)

station_info = config["Station_metadata"]
forcing_data = config["Forcing_data"]
station_data = {
    'lat': config["Station_metadata"]["lat"],
    'lon': config["Station_metadata"]["lon"],
    'elev': config["Station_metadata"]["elev"],
    'height_T': forcing_data["height_T"],
    'height_V': forcing_data["height_V"]}

forcing_format = forcing_data["forcing_format"]
start_date = pd.to_datetime(forcing_data["run_start"], utc=True)
end_date = pd.to_datetime(forcing_data["run_end"], utc=True)

# Surfex vegetation types : 
#1: no vegetation (smooth) - NO    2: no vegetation (rocks) - ROCK    3: permanent snow and ice - SNOW
#4: temperate broadleaf cold-deciduous summergreen - TEBD    5: boreal needleleaf evergreen - BONE
#6: tropical broadleaf evergreen - EVER    7: C3 cultures types - C3    8: C4 cultures types - C4
#9: irrigated crops - IRR10: grassland (C3) - GRAS     11: tropical grassland (C4) - TROG
#12: peat bogs, parks and gardens (irrigated grass) - PARK     13: tropical broadleaf deciduous - TRBD
#14: temperate broadleaf evergreen - TEBE     15: temperate needleleaf evergreen - TENE
#16: boreal broadleaf cold-deciduous summergreen - BOBD    17: boreal needleleaf cold-deciduous summergreen - BOND
#18: boreal grass - BOGR    19: shrub - SHRB

In [4]:
###### OSVAS ###################################
###### ( OFFLINE SURFEX VALIDATION SYSTEM)######
###### STEP 1.3: KMNI AUTHENTICATION ###########

# More info : https://developer.dataplatform.knmi.nl/open-data-api#

# There are several ways to authenticate yourself into ICOS
# In the example below, the temporal API token is used, which is available 
# at the bottom of https://cpauth.icos-cp.eu/home/ after you authenticate
# into the portal. The token lasts for 100.000 seconds, ~28 hours.

key_path = os.path.join(OSVAS,"KMNI_token.txt") #new file with KMNI credential (maybe modify cookie_ICOS to store all the keys for diferent services?)
api_key = open(key_path, "r").readline().strip()
endpoint = forcing_data['dataset1']['doi']
get_file_response = requests.get(endpoint, headers={"Authorization": api_key})
get_file_response.raise_for_status()

In [5]:
###### OSVAS #####################################################
###### ( OFFLINE SURFEX VALIDATION SYSTEM)########################
###### STEP 1.4: LOAD STATION DATA, READ VARIABLES ###############
###### TRANSFORM TO UNITS USED BY SURFEX #########################

datasets = {k: v for k, v in forcing_data.items() if k.startswith("dataset") or k.startswith("dataset_")}
common_timedelta = min([ds_info["timedelta"] for ds_name, ds_info in datasets.items()])
forcing_dataset = initialize_forcing_dataset(start_date, end_date, common_timedelta, station_data)
for ds_name, ds_info in datasets.items():
    print(f"Processing {ds_name} from DOI: {ds_info['doi']}")
    doi = ds_info["doi"]
    variable_map_raw = ds_info["variables"]
    forcing_src = retrieve_data(doi, start_date, end_date, api_key).interp(time = forcing_dataset['time'])#, kwargs={"fill_value": "extrapolate"})
    for target_var, entry in variable_map_raw.items():
        src, op, val = parse_variable_entry(entry, common_timedelta)
        try:
            forcing_var = list(forcing_dataset.filter_by_attrs(ascii_name=target_var).data_vars.keys())[0]
        except:
            continue
        attrs = forcing_dataset[forcing_var].attrs
        if src is None:
            if op == "const":
                forcing_dataset[forcing_var].values[:] = val
            else:
                raise ValueError(f"Unknown operation {op} for {target_var}")
        else:
            transformed = apply_transformation(forcing_src[src], op, val).values
            forcing_dataset[forcing_var].values[:] = np.transpose(np.array([transformed,]))

#Precipitation type splitting: Snow bellow 0ºC
forcing_dataset['Snowf'].values[:] = xr.where(forcing_dataset['Tair'] < 273.15, forcing_dataset['Rainf'], 0).values
forcing_dataset['Rainf'].values[:] = xr.where(forcing_dataset['Tair'] >= 273.15, forcing_dataset['Rainf'], 0).values

#Set negative radiations to 0
forcing_dataset['LWdown'] = forcing_dataset['LWdown'].clip(min = 0)
forcing_dataset['DIR_SWdown'] = forcing_dataset['DIR_SWdown'].clip(min = 0)

# Fill Gaps (Gaps here are whole files (months) missing, consider another period if this is necessary)
forcing_dataset = forcing_dataset.ffill(dim='time').bfill(dim='time')

Processing dataset1 from DOI: https://api.dataplatform.knmi.nl/open-data/v1/datasets/cesar_surface_meteo_lc1_t10/versions/v1.0/files/


36 files will be processed


Processing dataset2 from DOI: https://api.dataplatform.knmi.nl/open-data/v1/datasets/cesar_surface_radiation_lc1_t10/versions/v1.0/files/


36 files will be processed


In [6]:
###### OSVAS ####################################################
###### ( OFFLINE SURFEX VALIDATION SYSTEM)#######################
#### STEP 1.5: WRITE THE FORCING FILES IN THE SELECTED FILE #####
#### TYPE #######################################################

forcing_path=os.path.join(OSVAS,'forcings',Station_name)
write_forcing='yes' #Set to yes for writing forcing

if forcing_format == 'netcdf':
    os.makedirs(forcing_path, exist_ok=True)
    forcing_dataset.to_netcdf(os.path.join(OSVAS,'forcings',Station_name, 'FORCING.nc'))
elif forcing_format == 'ascii':
    os.makedirs(forcing_path, exist_ok=True)
    write_params_config(forcing_path, forcing_dataset)
    write_forcing_ascii(forcing_path, forcing_dataset)
else:
    raise ValueError(f"Invalid forcing format: {forcing_format}")